---
### 1. Necessary imports

In [1]:
import json
import os
import sys
from datetime import datetime, timezone

from graphiti_core import Graphiti
from graphiti_core.cross_encoder.gemini_reranker_client import GeminiRerankerClient
from graphiti_core.driver.falkordb_driver import FalkorDriver
from graphiti_core.embedder.gemini import GeminiEmbedder, GeminiEmbedderConfig
from graphiti_core.llm_client import LLMConfig
from graphiti_core.llm_client.gemini_client import GeminiClient
from graphiti_core.nodes import EpisodeType

In [2]:
# Allow imports from parent directory
sys.path.append(os.path.abspath(".."))

import envy

---
### 2. Connect to FalkorDB & Set up Graphiti indices

In [3]:
falkor_driver = FalkorDriver(**envy.FALKORDB_CREDENTIALS)
graphiti = Graphiti(
    graph_driver=falkor_driver,
    llm_client=GeminiClient(
        config=LLMConfig(api_key=envy.GOOGLE_API_KEY, model="gemini-gemini-2.5-flash")
    ),
    embedder=GeminiEmbedder(
        config=GeminiEmbedderConfig(
            api_key=envy.GOOGLE_API_KEY, embedding_model="embedding-001"
        )
    ),
    cross_encoder=GeminiRerankerClient(
        config=LLMConfig(api_key=envy.GOOGLE_API_KEY, model="gemini-gemini-2.5-pro")
    ),
)

---
### 3. Adding episodes

In [4]:
episodes = [
    {
        "content": "Kamala Harris is the Attorney General of California. She was previously "
        "the district attorney for San Francisco.",
        "type": EpisodeType.text,
        "description": "podcast transcript",
    },
    {
        "content": "As AG, Harris was in office from January 3, 2011 – January 3, 2017",
        "type": EpisodeType.text,
        "description": "podcast transcript",
    },
    {
        "content": {
            "name": "Gavin Newsom",
            "position": "Governor",
            "state": "California",
            "previous_role": "Lieutenant Governor",
            "previous_location": "San Francisco",
        },
        "type": EpisodeType.json,
        "description": "podcast metadata",
    },
    {
        "content": {
            "name": "Gavin Newsom",
            "position": "Governor",
            "term_start": "January 7, 2019",
            "term_end": "Present",
        },
        "type": EpisodeType.json,
        "description": "podcast metadata",
    },
]

In [5]:
# Add episodes to the graph
for i, episode in enumerate(episodes):
    await graphiti.add_episode(
        name=f"Freakonomics Radio {i}",
        episode_body=episode["content"]
        if isinstance(episode["content"], str)
        else json.dumps(episode["content"]),
        source=episode["type"],
        source_description=episode["description"],
        reference_time=datetime.now(timezone.utc),
    )
    print(f"Added episode: Freakonomics Radio {i} ({episode['type'].value})")

Added episode: Freakonomics Radio 0 (text)
Added episode: Freakonomics Radio 1 (text)
Added episode: Freakonomics Radio 2 (json)
Added episode: Freakonomics Radio 3 (json)


---
###  4. Node search: basic & recipes...

In [4]:
# Perform a hybrid search combining semantic similarity and BM25 retrieval
print("\nSearching for: 'Who was the California Attorney General?'")
results = await graphiti.search("Who was the California Attorney General?")

# Print search results
print("\nSearch Results:")
for result in results:
    print(f"UUID: {result.uuid}")
    print(f"Fact: {result.fact}")
    if hasattr(result, "valid_at") and result.valid_at:
        print(f"Valid from: {result.valid_at}")
    if hasattr(result, "invalid_at") and result.invalid_at:
        print(f"Valid until: {result.invalid_at}")
    print("---")


Searching for: 'Who was the California Attorney General?'

Search Results:
UUID: 589e17a0-36aa-46a0-89b2-2a932c7c69df
Fact: Kamala Harris is the Attorney General of California.
Valid from: 2025-11-29 13:17:46.110576+00:00
---
UUID: 2f5aa414-3652-4e2d-b0a2-c659e789fac5
Fact: Kamala Harris was previously the district attorney for San Francisco.
---
UUID: 72ee1337-c602-4f52-9cc9-1a1812d427f5
Fact: Gavin Newsom is the Governor of California.
Valid from: 2025-11-29 13:18:13.840262+00:00
---
UUID: b03adbfb-8024-4624-990f-66e7dc20b49a
Fact: Harris was in office as AG
Valid from: 2011-01-03 00:00:00+00:00
Valid until: 2017-01-03 00:00:00+00:00
---
UUID: bad23752-3848-4e98-9dea-d406d6fe693b
Fact: Gavin Newsom currently holds the position of Governor.
Valid from: 2025-11-29 13:18:13.840262+00:00
---
UUID: ca153ffb-8d34-44aa-ac56-1e4122dcbc3e
Fact: Gavin Newsom previously held the role of Lieutenant Governor.
---
UUID: e708da46-ce27-4049-8508-82649fe5b224
Fact: Gavin Newsom was previously locate